## Colab session configuration

## Reading hg38 genome assembly 

Set parent directory.

In [8]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import glob
import pathlib
from pathlib import Path
!pip install pyfaidx
from pyfaidx import Fasta

# Set data directory and base directory
base_dir = Path.cwd()

Download the hg38 fasta zipped file and unzip to standard fasta file

In [4]:
!mkdir -p data
!curl -L -C - "https://hgdownload.cse.ucsc.edu/goldenPath/hg38/bigZips/hg38.fa.gz" -o data/hg38.fa.gz
!curl -L -C - "https://hgdownload.cse.ucsc.edu/goldenPath/hg38/bigZips/hg38.chrom.sizes" -o data/hg38.chrom.sizes
!gunzip -c data/hg38.fa.gz > data/hg38.fa

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  938M  100  938M    0     0  63.9M      0  0:00:14  0:00:14 --:--:-- 60.9M
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 11672  100 11672    0     0  93291      0 --:--:-- --:--:-- --:--:-- 94129


Function to sample random sections of the genome, weighted by chromosome length and within the specified sequence length range.

In [9]:
# Set seed
rng = np.random.default_rng(111)

def sample_from_fasta(fasta_file, chrom_size_file, n_seqs, min_length, max_length):
    """
    Function to sample randomly from .fa genome

    Args:
        fasta_file      : Path to .fa file containing genome
        chrom_size_file : Path to tab-separate text file containing sequence names and sizes
        n_seqs          : Number of sequences to sample
        min_length      : Minimum sequence length to sample
        max_length      : Maximum sequence length to sample
    
    Returns:
        seqs : List of sampled sequences as strings
    """
    # Store file using pyfaidx
    f = Fasta(fasta_file)

    # Load the chromosome and sort
    size_df = pd.read_table(chrom_size_file, sep = "\t", index_col = 0, header = None)
    size_df.columns = ["length"]
    size_df["length"] = size_df["length"].astype(int)
    size_df = size_df.iloc[size_df.index.argsort()]

    # Check that the fasta chromsomes match the size df
    index = sorted(size_df.index.to_numpy())
    chromosomes = sorted(np.array(list(f.keys())))

    if index != chromosomes:
        raise KeyError("Fasta chromosomes IDs and length data IDs are incompatible")

    # Normalize to obtain sampling weights for each chromosome
    weights = size_df["length"].to_numpy() / size_df["length"].to_numpy().sum()

    # Initiate list to store sequences
    seqs = []

    # Counter so that rejected sequences don't count toward total count
    counter = 0

    # Number of sequences to sample
    while counter < n_seqs:

        # Weighted sampling of chromosome
        chrom = rng.choice(chromosomes, p = weights)
        length = int(size_df.loc[chrom, "length"])

        # Randomly sample a sequence
        start_idx = rng.integers(0, length)
        seq_length = rng.integers(min_length, max_length + 1)

        # If the sampled sequence is out of range
        if start_idx + seq_length > length:
            continue
        
        seq = f[chrom][start_idx:start_idx + seq_length]
        seq = str(seq)

        # Split across "N" character if exists and take first substr
        seq = seq.split("N")[0]

        # Convert to uppercase, remove non-alpha and non-ACGT characters 
        seq = "".join(char.strip().upper() for char in seq if char.isalpha() and char in "ACGTacgt")

        # Check that sequences is still above minimum length
        if len(seq) >= min_length:
            counter += 1 
            seqs.append(seq)

    return seqs

Sample from genome to collect pre-training data.

In [11]:
# Data directory
data_dir = base_dir / "data"
fasta_file = data_dir / "hg38.fa"
chrom_size_file = data_dir / "hg38.chrom.sizes"

# Set parameters
n_seqs = 1000
min_length = 20
max_length = 500

# Get pretraining data
pretraining = sample_from_fasta(
    fasta_file = fasta_file,
    chrom_size_file = chrom_size_file,
    n_seqs = n_seqs,
    min_length = min_length,
    max_length = max_length
)

print(pretraining)

['GTCTCAAACTTCTGGGCTCAAGTGGTCCTCCCACCTTGGCCTCCCAAAGTGCTAGAATTACAGGCAAGAGCCACTGTGCCCAGCCCACTAGCAATTTTTATATGGCAGTGGTTATGTCATCTTTGGTGCATGGACCTGATATCACATATTTTGGTTATTTAGGCTTTATTTGTTTACATCCTGGTTTTTTTTACCATGTTTCAAAGTATTGAGTTCATGATATAAAGAATATGTTATAAATGGCTCCTCAAATGGACTTAATTGAAATCCTTGGCACCATCATACTGCATCTGTCAACAACCTCCTCTGCCTAGTTAGCTCCAACTCATTTGTTAAGGCTCAGGACAAGAGTTATCTTCTTTAGGAAAACATACTTAATTTTGGTAACCCACTCTATTAAT', 'CAGGAGCTCTCCTCTCTCCATCCAACCTGGTTTAGGTTTCTGCCTAACCCCCTCACCTTTCTTGCAACCCTCTCAACTGACATCATTTGTCTTGGTGGGGTTCCCCACAACTCTGTAGTTCTTCTAGAGGAGTGACTGTGTCTTTTTACCTTTTTATCTCCAGCATTTGATAAATGGTGAATGGATGAATGAGACAGAGGCATGAATCTTTTGCAAAAAATAACATGACATCAAAATATATAGCAAAAAAACTATAACAAGTGAAAGTAGAAATGACAGTTATACAATCATGAGATATTTTAACACACTACTATCCATCTTTGAAAAATCAATTGAATAAAAAATAAATAAGGGCAAGGAAGAATTGAACTATATAACAAAGTTACATTAATAAATACTGAACTTTGTAACCCAGATATAGAAA', 'TCAGGCAGAAAAAAGAAATACAAGGTATCCAGATGGAAAGGAAGAGGTAAAACTGTCATATTCCCAGAGGACATGATAGTCAAGAATATTTTCTCTAAGAAAATCTAATGGAATCTATGAAAAACTCCACTAGTTTTGGTAAGTGGGTTTAGCAAAGTTCTAGGA

## Tokenizer

## Model

In [ ]:
!pip install accelerate
!pip install datasets
!pip install transformers
!pip install torch
!pip install flash-attn

import accelerate
import flash_attn
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
from datasets import load_dataset
from transformers import (
    AutoConfig,
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)

torch.backends.cudnn.benchmark=True

Implement a sparse attention transformer.

Implement the model with 10 transformer blocks.

In [ ]:
class miniglm(nn.Module):

    def __init__(self):